# Exploration du lexique : structuration des entrées et construction du corpus

**Projet** : corpus arabe tchadien latinisé ↔ français (code ISO 639-3 `shu`).
**Source** : *Lexique arabe tchadien – français*, SIL Tchad, 11e édition, 2021 (cf. `docs/01-methodologie.md`, section 1.3).

**Statut du notebook.** Version historique (v0.1) du traitement, conservée pour la
traçabilité. Le traitement de référence est `td-arabic build-corpus`
(`src/td_arabic/lexicon/corpus.py`, cf. `docs/01-methodologie.md`, section 1.6). Ce notebook
en documente chaque étape, permet d'examiner les cas difficiles et vérifie que ses
résultats concordent avec ceux du pipeline (étape 13).

**Il n'écrit jamais dans `data/processed/`** : toutes les sorties vont dans
`data/interim/notebook_v01/`.

## Les étapes

| # | Étape | Question de recherche | Sortie |
| --- | --- | --- | --- |
| 1 | Chargement | Les données d'entrée sont-elles celles attendues ? | `df` |
| 2 | Vedettes homographes | Comment distinguer `abad¹` de `abad²` ? | `headword` normalisée |
| 3 | Ligatures `fl` | Reste-t-il des `)` à la place de `fl` ? | `headword`, `forms` corrigées |
| 4 | Vérifications ciblées | Les cas connus sont-ils corrects ? | contrôle visuel |
| 5 | Détection des expressions | Quelles définitions contiennent `expr.` ? | `nb_expressions` |
| 6 | Règles manuelles | Quelle est la vérité terrain du découpage ? | `MANUAL_EXPRESSIONS` |
| 7 | Extraction automatique | Peut-on découper sans règle manuelle ? | précision mesurée |
| 8 | Structuration | Une ligne par entrée et par expression | `df_structure` |
| 9 | Contrôles | Les cas importants sont-ils exacts ? | assertions |
| 10 | Fusion | Lexique final dans l'ordre du PDF | `df_final` |
| 11 | Corpus ar → fr | Paires `src` → `output` | `corpus` |
| 12 | Corpus bilingue | Les deux directions | `corpus_bilingual` |
| 13 | Comparaison avec le pipeline | Les résultats concordent-ils ? | écarts attendus |

## Ce qui change par rapport à la première version du notebook

| Point | Avant | Maintenant | Effet sur les données |
| --- | --- | --- | --- |
| Chemins | `data/lexique/` (dossier disparu) | entrée `data/interim/`, sorties `data/interim/notebook_v01/` | aucun |
| Correction `)` → `fl` | appliquée à toute la ligne | limitée à `headword` et `forms` | aucun sur ces données |
| Homographes | `1` et `2` traités, le `3` laissé tel quel sans avertissement | idem, mais les cas non traités sont **signalés** (étape 2) | aucun |
| Effectif affiché à l'étape 8 | 4 832 (lexique entier) | 66 (lignes réellement traitées) | affichage seulement |
| Extracteur automatique | la traduction contenait la vedette suivante ; vedette cherchée avec le marqueur `ʾ` | découpage corrigé, vedette sans marqueur, **précision mesurée**, tout résultat automatique marqué à vérifier | aucun : les 66 entrées ont une règle manuelle |
| Ordre de `lexique_final` | entrées avec expressions rejetées à la fin | ordre du PDF conservé (expressions après leur entrée) | ordre des lignes |
| Colonnes de `lexique_final` | `type` vide hors expressions | `type` toujours renseigné, `parent_headword` ajouté | colonnes |
| `sense_number` | décimal (`2.0`) | entier (`2`) | format |
| Contrôles | affichages | assertions qui arrêtent le notebook en cas d'écart | aucun |

**Inchangé** : les 77 expressions validées à la main (recopiées telles quelles), le
marqueur `ʾ` des homographes de rang 2, et le corpus final de 4 909 paires.

## 0. Configuration

Tous les chemins et paramètres sont définis ici, pour que le reste du notebook ne contienne
aucune valeur en dur. L'empreinte SHA-256 du fichier d'entrée et les versions des outils sont
affichées pour pouvoir **reproduire** une exécution précise.

In [1]:
import hashlib
import re
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 160)


def find_project_root(start: Path | None = None) -> Path:
    # Remonte l'arborescence jusqu'au dossier qui contient data/interim.
    # Le notebook fonctionne ainsi lancé depuis la racine ou depuis notebooks/.
    start = (start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "data" / "interim").is_dir():
            return folder
    raise FileNotFoundError("Racine du projet introuvable (dossier data/interim attendu).")


ROOT = find_project_root()
INPUT_ENTRIES = ROOT / "data" / "interim" / "lexique_entries.csv"
OUT_DIR = ROOT / "data" / "interim" / "notebook_v01"      # sorties de ce notebook
PROCESSED_DIR = ROOT / "data" / "processed"               # lecture seule (étape 13)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Colonnes attendues en entrée (sortie de l'extraction du PDF).
EXPECTED_COLUMNS = ["headword", "sense_number", "forms", "pos", "definition",
                    "section", "page", "column", "ligature_reconstructed"]

# Marqueur ajouté à la vedette selon le numéro d'homographe (voir étape 2).
HOMOGRAPH_MARKERS = {"1": "", "2": "ʾ"}

# Ligature « fl » mal encodée par le PDF : « ) » à l'intérieur d'un mot.
LIGATURE_FIX = (")", "fl")

# Marqueur d'expression dans les définitions.
EXPR_PATTERN = re.compile(r"\bexpr\.", flags=re.IGNORECASE)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


print("Racine         :", ROOT.name)
print("Entrée         :", INPUT_ENTRIES.relative_to(ROOT))
print("SHA-256        :", sha256(INPUT_ENTRIES))
print("Sorties        :", OUT_DIR.relative_to(ROOT))
print("Python / pandas:", sys.version.split()[0], "/", pd.__version__)

Racine         : td_arabic
Entrée         : data/interim/lexique_entries.csv
SHA-256        : 28ca1d6374560ca35f5fb8347be97858219bc098b2e35b20205b114fe82e9d0c
Sorties        : data/interim/notebook_v01
Python / pandas: 3.11.15 / 3.0.5


## 1. Chargement

Le fichier `lexique_entries.csv` contient **une ligne par entrée du dictionnaire**, telle
qu'extraite du PDF (étape 1 du pipeline, sans OCR). On vérifie d'abord que sa structure est
celle attendue, car toute la suite en dépend.

- Le fichier actuel contient une colonne supplémentaire, `headword_base`, ajoutée depuis par
  le pipeline. Le notebook la recalcule lui-même (étape 2) puis vérifie qu'elle est identique.
- `sense_number` : numéro d'homographe (`abad¹`, `abad²`), vide pour les mots sans homographe.
  On le convertit en entier avec valeurs manquantes (`Int64`) plutôt qu'en décimal.
- `page` / `column` : localisation dans le PDF, indispensable pour revenir à la source
  quand un cas pose problème.

In [2]:
raw = pd.read_csv(INPUT_ENTRIES)

missing_columns = [c for c in EXPECTED_COLUMNS if c not in raw.columns]
assert not missing_columns, f"Colonnes manquantes : {missing_columns}"
print("Colonnes supplémentaires ignorées :", [c for c in raw.columns if c not in EXPECTED_COLUMNS])

df = raw[EXPECTED_COLUMNS].copy()
df["sense_number"] = df["sense_number"].astype("Int64")

print(f"{len(df)} entrées, pages {df['page'].min()} à {df['page'].max()}")
df.head()

Colonnes supplémentaires ignorées : ['headword_base']
4832 entrées, pages 15 à 171


,headword,sense_number,forms,pos,definition,section,page,column,ligature_reconstructed
0,aab,<NA>,inacc. yiʼiib,v.tr.,avoir honte,A a,15,left,False
1,aba,<NA>,inacc. yaaba / impr. aaba,v.tr.,refuser,A a,15,left,False
2,abad1,1,inacc. yaʼabud / impr. aʼabud,v.tr.,adorer Dieu,A a,15,left,False
3,abad2,2,NaN,n.m.,pour toujours ila l-abad expr. pour toujours,A a,15,left,False
4,abadan,<NA>,NaN,adv.,jamais,A a,15,left,False


In [3]:
# Valeurs manquantes par colonne : `forms` et `sense_number` sont naturellement souvent vides
# (beaucoup de mots n'ont ni forme fléchie documentée ni homographe).
df.isna().sum().to_frame("manquantes").assign(part=lambda t: (t["manquantes"] / len(df)).round(3))

,manquantes,part
headword,0,0.000
sense_number,4489,0.929
forms,1292,0.267
pos,1,0.000
definition,0,0.000
section,0,0.000
page,0,0.000
column,0,0.000
ligature_reconstructed,0,0.000


## 2. Vedettes homographes

Dans le PDF, deux mots de même graphie sont numérotés en exposant (`abad¹`, `abad²`). À
l'extraction, l'exposant devient un chiffre final (`abad1`, `abad2`).

La première version du traitement :

- supprime le `1` (premier sens, graphie « neutre ») ;
- remplace le `2` par le marqueur `ʾ` (`abadʾ`).

Ce marqueur sert ensuite de **clé** aux règles manuelles de l'étape 6 (`abadʾ`, `ahadʾ`, …).

> **Point d'attention.** Le numéro 3 existe aussi (8 entrées) et n'est pas traité par cette
> convention : il reste un chiffre dans la vedette (`adda3`). On ne l'invente pas ici (on
> n'a pas de convention validée pour un troisième homographe), mais on le **signale**.
> D'autre part `ʾ` (U+02BE) est aussi une variante de saisie de la coupure glottale
> `ʼ` (U+02BC, cf. `docs/02-corpus-datasheet.md`) : le marqueur peut donc se confondre avec
> une lettre. Le pipeline de référence évite les deux problèmes en gardant la vedette nue
> (`abad`) et en distinguant les homographes par `sense_number`.

Trois colonnes d'aide sont ajoutées pour la traçabilité : `headword_brut` (valeur du
fichier), `headword_base` (sans chiffre) et `homograph_digit`.

In [4]:
TRAILING_DIGITS = re.compile(r"(\d+)$")

df["headword_brut"] = df["headword"]
df["headword_base"] = df["headword"].str.replace(TRAILING_DIGITS, "", regex=True)
df["homograph_digit"] = df["headword"].str.extract(TRAILING_DIGITS)[0]

# Si le fichier fournit déjà `headword_base` (ajoutée par le pipeline), elle doit être identique.
if "headword_base" in raw.columns:
    assert (raw["headword_base"] == df["headword_base"]).all(), "headword_base du fichier ≠ recalcul"

# Contrôle de cohérence : un chiffre final n'existe que si `sense_number` est renseigné,
# et les deux valeurs sont identiques. Sinon l'extraction du PDF serait incohérente.
has_digit = df["homograph_digit"].notna()
assert has_digit.equals(df["sense_number"].notna()), "chiffre final et sense_number divergent"
assert (df.loc[has_digit, "homograph_digit"].astype(str)
        == df.loc[has_digit, "sense_number"].astype(str)).all()


def normalise_headword(base: str, digit) -> str:
    # Vedette sans chiffre + marqueur d'homographe ; un chiffre inconnu est conservé tel quel.
    if pd.isna(digit):
        return base
    return base + HOMOGRAPH_MARKERS.get(digit, digit)


df["headword"] = [normalise_headword(b, d) for b, d in zip(df["headword_base"], df["homograph_digit"])]

print("Répartition des numéros d'homographe :", df["homograph_digit"].value_counts().to_dict())
df.loc[has_digit, ["headword_brut", "headword", "sense_number", "pos", "definition"]].head(6)

Répartition des numéros d'homographe : {'2': 168, '1': 167, '3': 8}


,headword_brut,headword,sense_number,pos,definition
2,abad1,abad,1,v.tr.,adorer Dieu
3,abad2,abadʾ,2,n.m.,pour toujours ila l-abad expr. pour toujours
60,achmaan1,achmaan,1,adj.,confiant
61,achmaan2,achmaanʾ,2,adj.,méprisant
66,aada1,aada,1,v.tr.,contaminer
67,aada2,aadaʾ,2,v.tr.,devenir un ennemi


In [5]:
# Cas NON traités par la convention : le chiffre reste dans la vedette (limite connue).
unhandled = df[has_digit & ~df["homograph_digit"].isin(HOMOGRAPH_MARKERS)]
print(f"{len(unhandled)} vedettes avec un numéro d'homographe non géré : "
      f"{sorted(unhandled['homograph_digit'].unique())}")
unhandled[["headword", "sense_number", "pos", "definition", "page"]]

8 vedettes avec un numéro d'homographe non géré : ['3']


,headword,sense_number,pos,definition,page
79,adda3,3,v.tr.,compter,17
1238,darra3,3,v.tr.,vanner,53
1505,fazaʼ3,3,n.m.,secours,61
1780,hajja3,3,v.i.,accomplir un pélerinage,70
1987,himil3,3,n.m.,fardeau,76
2327,kaffar3,3,v.i.,blasphémer,87
4025,saag3,3,n.m.,tibia,146
4525,tchakka3,3,v.i.,vendre,161


In [6]:
# Les règles manuelles (étape 6) identifient une entrée par sa vedette : celle-ci doit donc
# être unique parmi les entrées concernées. On mesure ici l'unicité globale.
dupes = df[df["headword"].duplicated(keep=False)]
print(f"Vedettes en double après normalisation : {df['headword'].duplicated().sum()} "
      f"({dupes['headword'].nunique()} vedettes distinctes)")
dupes[["headword", "sense_number", "pos", "definition", "page"]].head(10)

Vedettes en double après normalisation : 0 (0 vedettes distinctes)


,headword,sense_number,pos,definition,page


## 3. Correction des ligatures `fl`

Le PDF utilise la ligature « fl » (police *CharisSIL Bold*), qui sort de l'extraction sous la
forme d'une parenthèse fermante au milieu d'un mot : `kha)a` pour `khafla` (« surprise »),
`ha)a` pour `hafla` (« festivités »), `ma)uul` pour `mafluul`.

Cette réparation est faite **en amont** par `pdf_extract.repair_ligatures` (colonne
`ligature_reconstructed`). On la garde ici comme filet de sécurité, limitée à `headword` et
`forms` : la version précédente l'appliquait à toute la ligne, ce qui aurait altéré une
définition contenant une vraie parenthèse (80 définitions en contiennent, par exemple
`rose (couleur)`).

In [7]:
old, new = LIGATURE_FIX
mask = df["headword"].str.contains(old, regex=False) | df["forms"].fillna("").str.contains(old, regex=False)
print(f"Entrées contenant encore « {old} » dans headword/forms : {mask.sum()}")
print(f"Entrées dont une ligature a déjà été reconstruite en amont : {df['ligature_reconstructed'].sum()}")
df.loc[mask, ["headword", "forms", "page", "column"]]

Entrées contenant encore « ) » dans headword/forms : 0
Entrées dont une ligature a déjà été reconstruite en amont : 209


,headword,forms,page,column


In [8]:
for column in ("headword", "forms"):
    df[column] = df[column].str.replace(old, new, regex=False)

assert not df["headword"].str.contains(old, regex=False).any()
print("Aucune ligature résiduelle dans les vedettes.")

Aucune ligature résiduelle dans les vedettes.


## 4. Vérifications ciblées

Contrôles sur des cas connus. On les désigne **par leur vedette** et non par leur position
dans le tableau (la version précédente utilisait `iloc[2544:2546]`, qui se décale dès que
l'extraction change).

- `khafla` / `khaflaan` : vérifient la correction des ligatures (étape 3).
- `ahad` / `ahadʾ` : deux homographes dont le second (« testament ») contient des
  expressions (`al-Ahad al-Gadiim`, `al-Ahad al-Jadiid`) **à l'intérieur de sa définition**.
  C'est ce qui a motivé l'étude des expressions (étapes 5 à 8) : sans traitement, la
  définition de `ahadʾ` mêle le sens principal et deux expressions.

In [9]:
SPOT_CHECK = ["khafla", "khaflaan", "ahad", "ahadʾ"]
spot = df[df["headword"].isin(SPOT_CHECK)][["headword", "sense_number", "forms", "pos", "definition", "page"]]
assert set(spot["headword"]) == set(SPOT_CHECK), "un cas de contrôle a disparu du lexique"
spot

,headword,sense_number,forms,pos,definition,page
134,ahad,1,NaN,n.m.,dimanche,19
135,ahadʾ,2,pl. uhuud,n.m.,testament al-Ahad al-Gadiim expr. Ancien Testament al-Ahad al-Jadi...,19
2544,khafla,<NA>,NaN,n.f.,surprise,93
2545,khaflaan,<NA>,f. khaflaane / pl. khaflaaniin,adj.,surpris,93


## 5. Détection des expressions

Certaines définitions contiennent des **expressions figées** introduites par le marqueur
`expr.` :

```
ahadʾ   testament al-Ahad al-Gadiim expr. Ancien Testament al-Ahad al-Jadiid expr. Nouveau Testament
        └ sens principal ┘└ expression 1 ┘      └ traduction 1 ┘└ expression 2 ┘      └ traduction 2 ┘
```

Pour un corpus parallèle, chaque expression doit devenir **sa propre paire** (arabe →
français) et la définition de l'entrée ne doit garder que le sens principal.

On compte d'abord les marqueurs `expr.` par définition (`nb_expressions`). Le motif
`\bexpr\.` impose une frontière de mot : il ne se déclenche pas sur un mot qui se terminerait
par `expr.`.

In [10]:
# Les définitions manquantes deviennent des chaînes vides : sans cela, `.str.contains` renverrait
# des valeurs manquantes et les filtres ci-dessous échoueraient.
df["definition"] = df["definition"].fillna("").astype(str)


def count_expressions(text: str) -> int:
    # Nombre exact de marqueurs « expr. » dans une définition.
    return len(EXPR_PATTERN.findall(text))


df["nb_expressions"] = df["definition"].apply(count_expressions)

print("Entrées par nombre d'expressions :", df["nb_expressions"].value_counts().sort_index().to_dict())
df[df["nb_expressions"] >= 1][["headword", "definition", "nb_expressions"]].head(10)

Entrées par nombre d'expressions : {0: 4766, 1: 57, 2: 7, 3: 2}


,headword,definition,nb_expressions
3,abadʾ,pour toujours ila l-abad expr. pour toujours,1
32,abyad,blanc abyad karr expr. très blanc,1
135,ahadʾ,testament al-Ahad al-Gadiim expr. Ancien Testament al-Ahad al-Jadi...,2
158,ajiine,pâte de mil ajiine zarga expr. pâte brune caramélisée,1
182,aks,contraire be l-aks expr. au contraire,1
190,akhbach,gris akhbach tiliny expr. très gris,1
191,akhdar,vert akhdar sirij expr. très vert,1
457,angaara,fleurs d’oseille almi angaara expr. boisson rouge sucrée,1
590,aasar,bousculer aasir galbak expr. prends courage!,1
596,asfar,jaune asfar wak expr. très jaune,1


In [11]:
# Sous-ensemble à structurer. (Le fichier portait le nom « …_with_multiple_expressions » alors
# qu'il contient toutes les entrées à partir d'UNE expression : il est renommé.)
df_expr = df[df["nb_expressions"] >= 1].copy()
df_expr.to_csv(OUT_DIR / "lexique_entries_with_expressions.csv", index=False)
print(f"{len(df_expr)} entrées avec au moins une expression "
      f"({df_expr['nb_expressions'].sum()} expressions au total)")

66 entrées avec au moins une expression (77 expressions au total)


## 6. Règles manuelles : vérité terrain du découpage

Pour chaque entrée contenant `expr.`, on a validé **à la main** :

- la définition principale ;
- chaque expression et sa traduction française.

C'est la **vérité terrain** du découpage, et ce qui garantit la qualité des paires produites :
un découpage automatique de ces définitions est fragile (étape 7). Le dictionnaire est recopié
**sans modification** depuis la version précédente ; son successeur dans le pipeline est
`data/resources/expressions_manuelles.csv` (concordance vérifiée à l'étape 13).

Clé = vedette normalisée (avec le marqueur `ʾ` pour les homographes de rang 2).
Valeur = `{"main": définition principale, "expressions": [(expression, traduction), …]}`.

In [12]:
MANUAL_EXPRESSIONS = {
    "abadʾ": {
        "main": "pour toujours",
        "expressions": [
            ("ila l-abad", "pour toujours"),
        ],
    },
    "abyad": {
        "main": "blanc",
        "expressions": [
            ("abyad karr", "très blanc"),
        ],
    },
    "ahadʾ": {
        "main": "testament",
        "expressions": [
            ("al-Ahad al-Gadiim", "Ancien Testament"),
            ("al-Ahad al-Jadiid", "Nouveau Testament"),
        ],
    },
    "ajiine": {
        "main": "pâte de mil",
        "expressions": [
            ("ajiine zarga", "pâte brune caramélisée"),
        ],
    },
    "aks": {
        "main": "contraire",
        "expressions": [
            ("be l-aks", "au contraire"),
        ],
    },
    "akhbach": {
        "main": "gris",
        "expressions": [
            ("akhbach tiliny", "très gris"),
        ],
    },
    "akhdar": {
        "main": "vert",
        "expressions": [
            ("akhdar sirij", "très vert"),
        ],
    },
    "angaara": {
        "main": "fleurs d’oseille",
        "expressions": [
            ("almi angaara", "boisson rouge sucrée"),
        ],
    },
    "aasar": {
        "main": "bousculer",
        "expressions": [
            ("aasir galbak", "prends courage!"),
        ],
    },
    "asfar": {
        "main": "jaune",
        "expressions": [
            ("asfar wak", "très jaune"),
        ],
    },
    "aswad": {
        "main": "noir",
        "expressions": [
            ("aswad kurum", "très noir"),
        ],
    },
    "ayyi": {
        "main": "chaque",
        "expressions": [
            ("ayyi waahid", "chacun"),
        ],
    },
    "azrag": {
        "main": "noir",
        "expressions": [
            ("azrag kurum", "très noir"),
            ("azrag litt", "très noir"),
        ],
    },
    "baalʾ": {
        "main": "attention",
        "expressions": [
            ("khutt baalak", "fais attention!"),
        ],
    },
    "baarak": {
        "main": "bénir",
        "expressions": [
            ("baarakallah", "Que Dieu soit béni!"),
        ],
    },
    "baatil": {
        "main": "maigre",
        "expressions": [
            ("kalaam baatil", "des mots sans valeur, faux"),
        ],
    },
    "be": {
        "main": "avec",
        "expressions": [
            ("be da", "comme ça"),
        ],
    },
    "buʼud": {
        "main": "distance",
        "expressions": [
            ("in cha Allah", "s’il plaît à Dieu"),
        ],
    },
    "chaggiin": {
        "main": "fait de couper",
        "expressions": [
            ("chaggiin al-ladaay", "fête du premier repas préparé par la mariée"),
        ],
    },
    "chakk": {
        "main": "doute",
        "expressions": [
            ("bala chakk", "sans aucun doute"),
        ],
    },
    "chamsi": {
        "main": "solaire",
        "expressions": [
            ("taakha chamsiiya", "panneau solaire"),
        ],
    },
    "chig": {
        "main": "différent",
        "expressions": [
            ("chig chig", "complètement différent"),
        ],
    },
    "dimme": {
        "main": "meurtre",
        "expressions": [
            ("katal al-dimme", "commettre un meurtre"),
        ],
    },
    "fadul": {
        "main": "grâce à",
        "expressions": [
            ("min fadlak", "s’il te plaît"),
        ],
    },
    "gaʼad": {
        "main": "être là",
        "expressions": [
            ("gaʼad tihit", "s’asseoir"),
        ],
    },
    "gille": {
        "main": "manque",
        "expressions": [
            ("gillit al-adab", "manque de discipline"),
        ],
    },
    "hadara": {
        "main": "présence",
        "expressions": [
            ("hadarat Allah", "présence de Dieu"),
        ],
    },
    "hagg": {
        "main": "droit",
        "expressions": [
            ("huguug al-insaan", "droits de l'homme"),
        ],
    },
    "hamd": {
        "main": "louange",
        "expressions": [
            ("hamdulillah", "Que Dieu soit loué!"),
        ],
    },
    "hattaʾ": {
        "main": "jusqu’à",
        "expressions": [
            ("hatta kan", "même si"),
        ],
    },
    "hije": {
        "main": "histoire",
        "expressions": [
            ("hije gusaar", "devinette"),
            ("hije tawiil", "conte"),
        ],
    },
    "humaar": {
        "main": "âne",
        "expressions": [
            ("humaar rifaay", "mule"),
        ],
    },
    "iidʾ": {
        "main": "fête",
        "expressions": [
            ("yoom al-iid", "jour de fête"),
        ],
    },
    "Injiil": {
        "main": "Évangile",
        "expressions": [
            ("Injiil al-Masiih", "Nouveau Testament"),
        ],
    },
    "jaay": {
        "main": "de ce côté",
        "expressions": [
            ("jaay jaay", "çà et là"),
        ],
    },
    "kulla": {
        "main": "tout",
        "expressions": [
            ("kulla yoom", "tous les jours"),
            ("kulla wakit", "tout le temps"),
            ("kulla bakaan", "partout"),
        ],
    },
    "kulli": {
        "main": "chaque",
        "expressions": [
            ("kulli waahid", "chacun"),
        ],
    },
    "khachum": {
        "main": "bouche",
        "expressions": [
            ("khuchuum buyuut", "clan"),
        ],
    },
    "khanam": {
        "main": "moutons, chèvres",
        "expressions": [
            ("khanam al-buyud", "chèvres"),
            ("khanam al-zurug", "moutons"),
        ],
    },
    "khanigiin": {
        "main": "étranglement",
        "expressions": [
            ("khanigiin al-gamar", "éclipse de lune"),
        ],
    },
    "khaariji": {
        "main": "extérieur",
        "expressions": [
            ("Wazaart al-khaarijiiye", "Ministère des Affaires étrangères"),
        ],
    },
    "makhmi": {
        "main": "évanoui",
        "expressions": [
            ("al-Sabi al-Makhmi", "Orion (constellation)"),
        ],
    },
    "malya": {
        "main": "plénitude",
        "expressions": [
            ("malyit iid", "poignée"),
        ],
    },
    "marra": {
        "main": "fois",
        "expressions": [
            ("iddat marraat", "plusieurs fois"),
            ("marraat waahidiin", "parfois"),
        ],
    },
    "matar": {
        "main": "pluie",
        "expressions": [
            ("hajjaaz al-matar", "arc-en-ciel"),
        ],
    },
    "mukhaddas": {
        "main": "saint",
        "expressions": [
            ("al-Kitaab al-Mukhaddas", "Bible"),
        ],
    },
    "mukhlis": {
        "main": "sincère",
        "expressions": [
            ("galib mukhlis", "intègre, cœur sans partage"),
            ("niiye mukhlisa", "de tout son cœur"),
        ],
    },
    "mumkin": {
        "main": "peut-être",
        "expressions": [
            ("ma mumkin", "ce n’est pas possible"),
        ],
    },
    "nakham": {
        "main": "gencives",
        "expressions": [
            ("be nakham sunuuni", "de justesse"),
        ],
    },
    "nimle": {
        "main": "fourmis",
        "expressions": [
            ("marad al-nimle", "cancer de la peau"),
        ],
    },
    "rabbaat": {
        "main": "qui fait des nœuds",
        "expressions": [
            ("rabbaat al-duruub", "coupeur de route"),
        ],
    },
    "raas": {
        "main": "tête",
        "expressions": [
            ("raas gawi", "désobéissance"),
        ],
    },
    "sawaʾ": {
        "main": "le même",
        "expressions": [
            ("sawa sawa", "exactement le même"),
        ],
    },
    "subhaan": {
        "main": "exalté",
        "expressions": [
            ("subhaan Allah", "Que Dieu soit glorifié!"),
            ("subhaanahu", "Qu’Il soit glorifié!"),
        ],
    },
    "sukkar": {
        "main": "sucre",
        "expressions": [
            ("raas sukkar", "pain de sucre"),
        ],
    },
    "sulub": {
        "main": "rein",
        "expressions": [
            ("arbit sulbak", "prends courage!"),
        ],
    },
    "surʼa": {
        "main": "rapidité",
        "expressions": [
            ("be surʼa", "rapidement"),
        ],
    },
    "taluuʼ": {
        "main": "fait de se lever",
        "expressions": [
            ("taluuʼ al-harraay", "lever du soleil"),
        ],
    },
    "tawraat": {
        "main": "Torah",
        "expressions": [
            ("Tawraat Muusa", "Loi de Moïse"),
        ],
    },
    "usbaʼ": {
        "main": "doigt",
        "expressions": [
            ("usbaʼ hana rijil", "orteil"),
        ],
    },
    "waguuʼ": {
        "main": "chute",
        "expressions": [
            ("waguuʼ al-harraay", "coucher du soleil"),
        ],
    },
    "wajaʼ": {
        "main": "souffrance",
        "expressions": [
            ("wajaʼ een", "conjonctivite"),
            ("wajaʼ batun", "mal au ventre"),
            ("wajaʼ galib", "nausée"),
        ],
    },
    "waaluuda": {
        "main": "accouchement",
        "expressions": [
            ("beet al-waaluuda", "utérus"),
        ],
    },
    "wara": {
        "main": "derrière",
        "expressions": [
            ("wara beet", "cabinet, toilette"),
        ],
    },
    "yo": {
        "main": "oui",
        "expressions": [
            ("yo sameh", "d’accord"),
        ],
    },
    "Zabuur": {
        "main": "Psaumes",
        "expressions": [
            ("Kitaab al-Zabuur", "livre des Psaumes"),
        ],
    },
}

### 6.1 Cohérence des règles manuelles avec les données

Avant de s'y fier, on vérifie que les règles et le lexique se correspondent exactement :

1. chaque entrée contenant `expr.` a une règle ;
2. aucune règle ne vise une entrée inexistante (faute de frappe sur la vedette) ;
3. le nombre d'expressions de la règle égale le nombre de marqueurs `expr.` de la définition.

In [13]:
heads = df_expr["headword"]
assert heads.is_unique, "deux entrées à expressions partagent la même vedette : clé ambiguë"

missing_rule = sorted(set(heads) - set(MANUAL_EXPRESSIONS))
unused_rule = sorted(set(MANUAL_EXPRESSIONS) - set(heads))
count_mismatch = [
    (h, n, len(MANUAL_EXPRESSIONS[h]["expressions"]))
    for h, n in zip(heads, df_expr["nb_expressions"])
    if h in MANUAL_EXPRESSIONS and n != len(MANUAL_EXPRESSIONS[h]["expressions"])
]

print("Entrées sans règle manuelle           :", missing_rule)
print("Règles sans entrée correspondante     :", unused_rule)
print("Nombre de marqueurs ≠ nb d'expressions:", count_mismatch)
print(f"Couverture : {len(heads) - len(missing_rule)}/{len(heads)} entrées, "
      f"{sum(len(v['expressions']) for v in MANUAL_EXPRESSIONS.values())} expressions")

assert not missing_rule and not unused_rule and not count_mismatch

Entrées sans règle manuelle           : []
Règles sans entrée correspondante     : []
Nombre de marqueurs ≠ nb d'expressions: []
Couverture : 66/66 entrées, 77 expressions


## 7. Extraction automatique (secours et diagnostic)

Si une entrée **n'avait pas** de règle manuelle, on tenterait de la découper automatiquement.
L'idée : la vedette de l'expression contient en général la vedette de l'entrée (ou une de ses
formes) et se trouve juste avant le marqueur `expr.`.

```
[sens principal] [expression 1 expr.] [traduction 1] [expression 2 expr.] [traduction 2]
```

On découpe donc le texte aux marqueurs `expr.`, puis, dans chaque segment, on cherche la
vedette (ou une forme) : ce qui précède est la fin de la traduction précédente (ou le sens
principal pour le premier segment), ce qui suit est la vedette de l'expression.

Deux défauts de la version précédente sont corrigés :

- la traduction d'une expression **incluait la vedette de l'expression suivante** ;
- la vedette était cherchée **avec le marqueur `ʾ`** (`abadʾ`), donc jamais trouvée pour les
  homographes de rang 2.

Cette fonction n'est pas utilisée pour produire les données (toutes les entrées ont une règle
manuelle) mais elle est **évaluée** ci-dessous contre la vérité terrain.

In [14]:
FORM_PATTERN = re.compile(r"(?:f\.|pl\.|sg\.|dl\.|var\.)\s*([^/]+)", flags=re.IGNORECASE)


def clean_text(value) -> str:
    # Texte sans espaces multiples ni valeurs manquantes (None, NaN, <NA> → "").
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def search_candidates(headword_base: str, forms) -> list[str]:
    # Chaînes susceptibles d'ouvrir une expression : la vedette nue, puis ses formes
    # (féminin, pluriel, variantes). Sans doublon, dans cet ordre.
    found = [clean_text(headword_base)]
    found += [clean_text(item) for item in FORM_PATTERN.findall(clean_text(forms))]
    unique = []
    for item in found:
        if item and item not in unique:
            unique.append(item)
    return unique


def split_segment(segment: str, candidates: list[str]):
    # Sépare un segment en (texte avant, vedette d'expression). On retient la
    # première occurrence la plus à DROITE parmi les candidats, car la vedette est collée au
    # marqueur. Renvoie None si aucun candidat n'est trouvé.
    best = None
    for candidate in candidates:
        match = re.search(re.escape(candidate), segment, flags=re.IGNORECASE)
        if match and (best is None or match.start() > best.start()):
            best = match
    if best is None:
        return None
    return clean_text(segment[:best.start()]), clean_text(segment[best.start():])


def extract_expressions_automatic(headword_base: str, forms, definition: str):
    # Découpe automatique. Renvoie (expressions, définition principale, à_vérifier).
    definition = clean_text(definition)
    if not definition:
        return [], definition, True

    markers = list(EXPR_PATTERN.finditer(definition))
    if not markers:
        return [], definition, False

    candidates = search_candidates(headword_base, forms)
    starts = [0] + [m.end() for m in markers[:-1]]
    segments = [definition[start:m.start()] for start, m in zip(starts, markers)]
    tail = definition[markers[-1].end():]            # traduction de la dernière expression

    main_definition, headwords, translations = "", [], []
    review = False
    for i, segment in enumerate(segments):
        parts = split_segment(segment, candidates)
        if parts is None:                            # vedette introuvable : découpage douteux
            review = True
            continue
        before, expression_headword = parts
        if i == 0:
            main_definition = before
        else:
            translations.append(before)              # fin de la traduction précédente
        headwords.append(expression_headword)
    translations.append(clean_text(tail))

    expressions = [
        {"expression_number": n, "headword": h, "definition": t}
        for n, (h, t) in enumerate(zip(headwords, translations), start=1)
    ]
    return expressions, main_definition, review

### 7.1 Évaluation de l'extracteur automatique

Les 66 entrées ont une règle manuelle : on peut donc mesurer **à quel point** l'extraction
automatique retrouve la vérité terrain. Un découpage est jugé exact si la définition
principale, les vedettes d'expression **et** leurs traductions sont identiques, sans
signalement.

C'est un résultat méthodologique en soi : il justifie le choix de règles manuelles pour
un corpus destiné à entraîner des modèles.

In [15]:
evaluation = []
for row in df_expr.itertuples():
    truth = MANUAL_EXPRESSIONS[row.headword]
    expressions, main, review = extract_expressions_automatic(row.headword_base, row.forms, row.definition)
    predicted = [(e["headword"], e["definition"]) for e in expressions]
    evaluation.append({
        "headword": row.headword,
        "main_ok": main == truth["main"],
        "expressions_ok": predicted == truth["expressions"],
        "flagged": review,
        "exact": main == truth["main"] and predicted == truth["expressions"] and not review,
    })
evaluation = pd.DataFrame(evaluation)

print(f"Découpage exact          : {evaluation['exact'].sum()}/{len(evaluation)} "
      f"({evaluation['exact'].mean():.0%})")
print(f"Définition principale OK : {evaluation['main_ok'].sum()}/{len(evaluation)}")
print(f"Expressions OK           : {evaluation['expressions_ok'].sum()}/{len(evaluation)}")
print(f"Signalées (douteuses)    : {evaluation['flagged'].sum()}/{len(evaluation)}")

Découpage exact          : 36/66 (55%)
Définition principale OK : 36/66
Expressions OK           : 37/66
Signalées (douteuses)    : 6/66


In [16]:
# Pourquoi échoue-t-il ? On classe chaque échec, puis on affiche des exemples face à la vérité terrain.
def failure_category(row, flagged: bool, truth: dict) -> str:
    if flagged:
        return "signalé : vedette d'expression introuvable"
    base = row.headword_base.lower()
    # Toutes les expressions contiennent la vedette de l'entrée, mais précédée d'autres mots.
    if all(base in expr.lower() and expr.lower().find(base) > 0 for expr, _ in truth["expressions"]):
        return "mots avant la vedette de l'entrée"
    return "autre (coupure au mauvais endroit)"


errors = []
for row in df_expr.itertuples():
    expressions, main, review = extract_expressions_automatic(row.headword_base, row.forms, row.definition)
    truth = MANUAL_EXPRESSIONS[row.headword]
    predicted = [(e["headword"], e["definition"]) for e in expressions]
    if main != truth["main"] or predicted != truth["expressions"] or review:
        errors.append({"headword": row.headword, "cause": failure_category(row, review, truth),
                       "auto_main": main, "auto_expressions": predicted,
                       "truth_main": truth["main"], "truth_expressions": truth["expressions"]})
errors = pd.DataFrame(errors)

print(errors["cause"].value_counts().to_string(), end="\n\n")
errors.sort_values("cause").groupby("cause").head(2)[["headword", "cause", "auto_expressions", "truth_expressions"]]

cause
mots avant la vedette de l'entrée             22
signalé : vedette d'expression introuvable     6
autre (coupure au mauvais endroit)             2



,headword,cause,auto_expressions,truth_expressions
14,khachum,autre (coupure au mauvais endroit),"[(khuchuum buyuut, clan)]","[(khuchuum buyuut, clan)]"
18,marra,autre (coupure au mauvais endroit),"[(marraat, plusieurs fois), (marraat waahidiin, parfois)]","[(iddat marraat, plusieurs fois), (marraat waahidiin, parfois)]"
0,abadʾ,mots avant la vedette de l'entrée,"[(abad, pour toujours)]","[(ila l-abad, pour toujours)]"
1,ahadʾ,mots avant la vedette de l'entrée,"[(Ahad al-Gadiim, Ancien Testament al-), (Ahad al-Jadiid, Nouveau ...","[(al-Ahad al-Gadiim, Ancien Testament), (al-Ahad al-Jadiid, Nouvea..."
4,aasar,signalé : vedette d'expression introuvable,[],"[(aasir galbak, prends courage!)]"
7,buʼud,signalé : vedette d'expression introuvable,[],"[(in cha Allah, s’il plaît à Dieu)]"


**Lecture.** Trois causes d'échec apparaissent (effectifs ci-dessus) :

1. **Mots avant la vedette de l'entrée** : `be l-aks` (pour `aks`), `ila l-abad`, `khutt baalak`,
   `kalaam baatil`, `al-Ahad al-Gadiim`. La recherche trouve `aks` dans `be l-aks` et coupe
   après le `be l-` : ce préfixe reste collé au sens principal. Une règle purement textuelle
   ne peut pas deviner où commence l'expression ; il faudrait connaître les particules
   possibles (`be`, `ila`, `al-`, …) ou valider à la main.
2. **Vedette d'expression sans rapport direct avec l'entrée** : `aasir galbak` (pour
   `aasar`, forme conjuguée), `min fadlak` (pour `fadul`), `in cha Allah` (pour `buʼud`). Le
   cas est correctement **signalé** comme douteux.
3. **Coupure au mauvais endroit** : une chaîne recherchée apparaît à l'intérieur d'un autre mot
   (`marra` dans `marraat`) ou une forme fléchie ressemble à un mot courant.

**Conséquence dans ce notebook** : un résultat automatique n'est jamais considéré comme fiable.
Tout découpage obtenu sans règle manuelle est écrit dans le fichier de revue
(`lexique_expressions_review.csv`) pour validation humaine.

## 8. Structuration : une ligne par entrée et par expression

Pour chaque entrée contenant `expr.` on produit :

- **une ligne `entry`** avec la définition principale seule ;
- **une ligne `expression`** par expression, avec `forms = "expr."`, la même catégorie
  grammaticale, la même page que l'entrée, et `parent_headword` pour retrouver l'entrée.

Les règles manuelles sont prioritaires ; l'extraction automatique n'est qu'un secours, dont
tout résultat est envoyé en revue.

In [17]:
COLUMNS = ["headword", "sense_number", "forms", "pos", "definition", "section", "page", "column",
           "ligature_reconstructed", "nb_expressions", "type", "parent_headword", "expression_number"]


def extract_expressions(row):
    # Renvoie (expressions, définition principale, motifs de revue) pour une entrée.
    if row["headword"] in MANUAL_EXPRESSIONS:
        rule = MANUAL_EXPRESSIONS[row["headword"]]
        expressions = [
            {"expression_number": n, "headword": clean_text(h), "definition": clean_text(t)}
            for n, (h, t) in enumerate(rule["expressions"], start=1)
        ]
        return expressions, clean_text(rule["main"]), []

    expressions, main, uncertain = extract_expressions_automatic(
        row["headword_base"], row["forms"], row["definition"])
    reasons = ["Extraction automatique non validée (précision mesurée à l'étape 7.1)"]
    if uncertain or not expressions:
        reasons.append("Expression non identifiable automatiquement")
    return expressions, main, reasons


def structure_entry(entry_id, row):
    # Lignes (entrée + expressions) et lignes de revue pour une entrée contenant `expr.`.
    expressions, main, reasons = extract_expressions(row)
    if not expressions:                             # rien de découpé : on garde la définition d'origine
        main = row["definition"]

    shared = {key: row[key] for key in ("sense_number", "pos", "section", "page", "column",
                                        "ligature_reconstructed", "nb_expressions")}
    rows = [{"entry_id": entry_id, "headword": row["headword"], "headword_base": row["headword_base"],
             "forms": row["forms"],
             "definition": clean_text(main), "type": "entry",
             "parent_headword": "", "expression_number": "", **shared}]
    rows += [{"entry_id": entry_id, "headword": e["headword"], "headword_base": e["headword"],
              "forms": "expr.",
              "definition": e["definition"], "type": "expression",
              "parent_headword": row["headword"], "expression_number": e["expression_number"], **shared}
             for e in expressions]
    reviews = [{"headword": row["headword"], "definition_originale": row["definition"],
                "page": row["page"], "raison": reason} for reason in reasons]
    return rows, reviews


structure_rows, review_rows = [], []
for entry_id, row in df_expr.iterrows():           # entry_id = position dans le lexique d'origine
    rows, reviews = structure_entry(entry_id, row)
    structure_rows += rows
    review_rows += reviews

df_structure = pd.DataFrame(structure_rows)
df_structure["sense_number"] = df_structure["sense_number"].astype("Int64")

In [18]:
# Sauvegarde (encodage utf-8-sig pour une ouverture correcte dans un tableur) et statistiques.
df_structure[COLUMNS].to_csv(OUT_DIR / "lexique_expressions_structure.csv", index=False, encoding="utf-8-sig")

review_path = OUT_DIR / "lexique_expressions_review.csv"
if review_rows:
    pd.DataFrame(review_rows).to_csv(review_path, index=False, encoding="utf-8-sig")
elif review_path.exists():
    review_path.unlink()                            # pas de fichier périmé d'une exécution précédente

n_entries = (df_structure["type"] == "entry").sum()
n_expressions = (df_structure["type"] == "expression").sum()

print(f"Entrées traitées (avec expr.)   : {len(df_expr)}")      # (avant : 4 832, effectif du lexique entier)
print(f"Lignes « entry »                : {n_entries}")
print(f"Lignes « expression »           : {n_expressions}")
print(f"Lignes produites                : {len(df_structure)}")
print(f"Cas à vérifier (revue)          : {len(review_rows)}")

assert n_entries == len(df_expr)
assert n_expressions == df_expr["nb_expressions"].sum()

Entrées traitées (avec expr.)   : 66
Lignes « entry »                : 66
Lignes « expression »           : 77
Lignes produites                : 143
Cas à vérifier (revue)          : 0


In [19]:
df_structure[COLUMNS].head(10)

,headword,sense_number,forms,pos,definition,section,page,column,ligature_reconstructed,nb_expressions,type,parent_headword,expression_number
0,abadʾ,2,NaN,n.m.,pour toujours,A a,15,left,False,1,entry,,
1,ila l-abad,2,expr.,n.m.,pour toujours,A a,15,left,False,1,expression,abadʾ,1
2,abyad,<NA>,f. beeda / pl. buyud,adj.,blanc,A a,16,left,False,1,entry,,
3,abyad karr,<NA>,expr.,adj.,très blanc,A a,16,left,False,1,expression,abyad,1
4,ahadʾ,2,pl. uhuud,n.m.,testament,A a,19,left,False,2,entry,,
5,al-Ahad al-Gadiim,2,expr.,n.m.,Ancien Testament,A a,19,left,False,2,expression,ahadʾ,1
6,al-Ahad al-Jadiid,2,expr.,n.m.,Nouveau Testament,A a,19,left,False,2,expression,ahadʾ,2
7,ajiine,<NA>,pl. ajaayin,n.f.,pâte de mil,A a,20,left,False,1,entry,,
8,ajiine zarga,<NA>,expr.,n.f.,pâte brune caramélisée,A a,20,left,False,1,expression,ajiine,1
9,aks,<NA>,NaN,adj.,contraire,A a,20,right,False,1,entry,,


## 9. Contrôles des cas importants

On vérifie des cas représentatifs (une expression, plusieurs, un homographe, une locution
courante) et on **arrête le notebook** s'ils sont faux, au lieu de se contenter de les afficher.

In [20]:
def show_family(parent: str) -> pd.DataFrame:
    # Entrée et expressions qui en dépendent.
    mask = (df_structure["headword"] == parent) | (df_structure["parent_headword"] == parent)
    return df_structure.loc[mask, ["headword", "forms", "definition", "type", "expression_number"]]


CHECK_HEADWORDS = ["abadʾ", "ahadʾ", "aks", "kulla", "wajaʼ", "marra", "hije", "subhaan"]
for word in CHECK_HEADWORDS:
    print(f"[{word}]")
    print(show_family(word).to_string(index=False), end="\n\n")

[abadʾ]
  headword forms    definition       type expression_number
     abadʾ   NaN pour toujours      entry                  
ila l-abad expr. pour toujours expression                 1

[ahadʾ]
         headword     forms        definition       type expression_number
            ahadʾ pl. uhuud         testament      entry                  
al-Ahad al-Gadiim     expr.  Ancien Testament expression                 1
al-Ahad al-Jadiid     expr. Nouveau Testament expression                 2

[aks]
headword forms   definition       type expression_number
     aks   NaN    contraire      entry                  
be l-aks expr. au contraire expression                 1

[kulla]
    headword forms     definition       type expression_number
       kulla   NaN           tout      entry                  
  kulla yoom expr. tous les jours expression                 1
 kulla wakit expr.  tout le temps expression                 2
kulla bakaan expr.        partout expression                 3



In [21]:
# (entrée, expression, traduction attendue, numéro) : cas relus à la main dans le lexique.
EXPECTED_CASES = [
    ("aks", "be l-aks", "au contraire", 1),
    ("abadʾ", "ila l-abad", "pour toujours", 1),
    ("ahadʾ", "al-Ahad al-Jadiid", "Nouveau Testament", 2),
    ("kulla", "kulla bakaan", "partout", 3),
    ("wajaʼ", "wajaʼ galib", "nausée", 3),
]
for parent, expression, translation, number in EXPECTED_CASES:
    found = df_structure[(df_structure["parent_headword"] == parent)
                         & (df_structure["headword"] == expression)
                         & (df_structure["definition"] == translation)
                         & (df_structure["forms"] == "expr.")
                         & (df_structure["expression_number"] == number)]
    assert len(found) == 1, f"cas incorrect ou introuvable : {parent} -> {expression}"
print(f"{len(EXPECTED_CASES)}/{len(EXPECTED_CASES)} cas attendus vérifiés.")

5/5 cas attendus vérifiés.


## 10. Fusion : lexique final

On réunit :

- les entrées **sans** expression, inchangées ;
- les lignes structurées de l'étape 8 (entrée + expressions).

La première version concaténait les deux blocs, ce qui **rejetait les 66 entrées à
expressions à la fin du fichier**, hors de leur place alphabétique. On les remet ici à leur
position d'origine (l'expression juste après son entrée), grâce à `entry_id` : l'ordre du
dictionnaire est ainsi conservé, comme dans le pipeline.

In [22]:
df_plain = df[df["nb_expressions"] == 0].copy()
df_plain["entry_id"] = df_plain.index
df_plain["type"] = "entry"
df_plain["parent_headword"] = ""
df_plain["expression_number"] = ""

df_final = pd.concat([df_plain, df_structure], ignore_index=True)

# Tri stable : entrée d'abord (numéro d'expression 0), puis ses expressions dans l'ordre.
df_final["_order"] = pd.to_numeric(df_final["expression_number"], errors="coerce").fillna(0)
df_final = (df_final.sort_values(["entry_id", "_order"], kind="stable")
                    .reset_index(drop=True))

# Aucune ligne perdue ni dupliquée : 4 832 − 66 entrées à expressions + 143 lignes structurées.
assert len(df_final) == len(df) - len(df_expr) + len(df_structure)
assert df_final["entry_id"].nunique() == len(df)

df_final[COLUMNS].to_csv(OUT_DIR / "lexique_final.csv", index=False)
print(f"{len(df)} entrées d'origine → {len(df_final)} lignes "
      f"({(df_final['type'] == 'entry').sum()} entrées + {(df_final['type'] == 'expression').sum()} expressions)")
df_final.loc[df_final["headword"].isin(["ahadʾ", "al-Ahad al-Gadiim", "al-Ahad al-Jadiid"]), COLUMNS]

4832 entrées d'origine → 4909 lignes (4832 entrées + 77 expressions)


,headword,sense_number,forms,pos,definition,section,page,column,ligature_reconstructed,nb_expressions,type,parent_headword,expression_number
137,ahadʾ,2,pl. uhuud,n.m.,testament,A a,19,left,False,2,entry,,
138,al-Ahad al-Gadiim,2,expr.,n.m.,Ancien Testament,A a,19,left,False,2,expression,ahadʾ,1
139,al-Ahad al-Jadiid,2,expr.,n.m.,Nouveau Testament,A a,19,left,False,2,expression,ahadʾ,2


## 11. Corpus arabe tchadien → français

Chaque ligne du lexique devient une paire d'entraînement :

- `src` : la vedette (arabe tchadien latinisé) ;
- `output` : sa définition française ;
- `pos`, `forms`, `sense_number` sont conservés comme métadonnées.

On retire les paires dont la source ou la traduction est vide, après suppression des espaces
de bord.

**Doublons.** La première version contenait une déduplication sur `(src, output)` laissée en
commentaire. On ne supprime rien : on **mesure** les paires `(src, output)` identiques. Elles
seraient légitimes (sens ou catégories différents d'un même mot) et la décision de
dédupliquer dépend de la tâche d'entraînement. Un cas voisin existe déjà :
`abadʾ → pour toujours` et `ila l-abad → pour toujours` ont la même traduction mais des
sources différentes ; ce ne sont pas des doublons de paires, mais de cibles.

In [23]:
corpus = df_final[["headword", "pos", "definition", "forms", "sense_number"]].rename(
    columns={"headword": "src", "definition": "output"})

corpus = corpus.dropna(subset=["src", "output"]).copy()
corpus["src"] = corpus["src"].astype(str).str.strip()
corpus["output"] = corpus["output"].astype(str).str.strip()
corpus = corpus[(corpus["src"] != "") & (corpus["output"] != "")].reset_index(drop=True)

corpus.to_csv(OUT_DIR / "corpus_arabe_tchadien_francais.csv", index=False)

n_dupes = corpus.duplicated(subset=["src", "output"]).sum()
n_same_target = corpus.duplicated(subset=["output"]).sum()
print(f"{len(corpus)} paires ; {n_dupes} paires (src, output) en double ; "
      f"{n_same_target} lignes dont la traduction apparaît déjà ailleurs")
corpus.head(10)

4909 paires ; 0 paires (src, output) en double ; 848 lignes dont la traduction apparaît déjà ailleurs


,src,pos,output,forms,sense_number
0,aab,v.tr.,avoir honte,inacc. yiʼiib,<NA>
1,aba,v.tr.,refuser,inacc. yaaba / impr. aaba,<NA>
2,abad,v.tr.,adorer Dieu,inacc. yaʼabud / impr. aʼabud,1
3,abadʾ,n.m.,pour toujours,NaN,2
4,ila l-abad,n.m.,pour toujours,expr.,2
5,abadan,adv.,jamais,NaN,<NA>
6,abadi,adj.,éternel,f. abadiiye,<NA>
7,abaat,n.m.,aisselle,pl. ubutte,<NA>
8,abba,n.m.,papa,NaN,<NA>
9,abbad,v.tr.,réduire en esclavage,inacc. yiʼabbid,<NA>


## 12. Corpus bilingue

On duplique chaque paire dans les deux directions (`ar-fr` et `fr-ar`) pour entraîner un
modèle de traduction dans les deux sens. La colonne `direction` l'indique.

> Dans les lignes `fr-ar`, `pos`, `forms` et `sense_number` décrivent toujours le mot
> **arabe**, qui devient ici la cible.

In [24]:
corpus_ar_fr = corpus.assign(direction="ar-fr")
corpus_fr_ar = corpus.rename(columns={"src": "output", "output": "src"}).assign(direction="fr-ar")
corpus_bilingual = pd.concat([corpus_ar_fr, corpus_fr_ar[corpus_ar_fr.columns]], ignore_index=True)

assert len(corpus_bilingual) == 2 * len(corpus)
corpus_bilingual.to_csv(OUT_DIR / "corpus_bilingue.csv", index=False)

print(corpus_bilingual["direction"].value_counts().to_dict())
corpus_bilingual.iloc[[0, 1, len(corpus), len(corpus) + 1]]

{'ar-fr': 4909, 'fr-ar': 4909}


,src,pos,output,forms,sense_number,direction
0,aab,v.tr.,avoir honte,inacc. yiʼiib,<NA>,ar-fr
1,aba,v.tr.,refuser,inacc. yaaba / impr. aaba,<NA>,ar-fr
4909,avoir honte,v.tr.,aab,inacc. yiʼiib,<NA>,fr-ar
4910,refuser,v.tr.,aba,inacc. yaaba / impr. aaba,<NA>,fr-ar


## 13. Comparaison avec le pipeline de référence

Dernier contrôle : les résultats de ce notebook concordent-ils avec `data/processed/`
(`td-arabic build-corpus`) ? Les deux traitements sont indépendants, donc leur accord est une
vérification croisée.

La comparaison est faite ligne à ligne (même ordre, même effectif) et porte sur la vedette
**nue** : le notebook ajoute `ʾ` aux homographes de rang 2 et garde le chiffre du rang 3 (étape 2),
alors que le pipeline garde la vedette sans marqueur ; ces marqueurs sont donc retirés avant
de comparer.

**Seul écart attendu** : `pos`, car le pipeline harmonise les variantes typographiques
(`n.f` → `n.f.`). Toute autre différence est une anomalie à examiner.

In [25]:
prod = pd.read_csv(PROCESSED_DIR / "lexique_final.csv", dtype=str, keep_default_na=False)

print(f"Lignes : notebook {len(df_final)} / pipeline {len(prod)}")
assert len(prod) == len(df_final), "effectifs différents : comparaison ligne à ligne impossible"

def as_text(series: pd.Series) -> pd.Series:
    # Valeurs manquantes → "" (le fichier du pipeline est lu sans valeurs manquantes).
    return series.astype("string").fillna("")


mine = pd.DataFrame({column: as_text(df_final[column]) for column in COLUMNS})
mine["headword_nue"] = as_text(df_final["headword_base"])      # vedette sans marqueur ni chiffre

comparison = {
    "headword (sans marqueur)": (mine["headword_nue"] != prod["headword"]).sum(),
    "definition": (mine["definition"] != prod["definition"]).sum(),
    "forms": (mine["forms"] != prod["forms"]).sum(),
    "pos": (mine["pos"] != prod["pos"]).sum(),
    "sense_number": (mine["sense_number"] != prod["sense_number"]).sum(),
    "type": (mine["type"] != prod["type"]).sum(),
    "page": (mine["page"] != prod["page"]).sum(),
    "expression_number": (mine["expression_number"] != prod["expression_number"]).sum(),
}
pd.Series(comparison, name="lignes différentes").to_frame()

Lignes : notebook 4909 / pipeline 4909


,lignes différentes
headword (sans marqueur),0
definition,0
forms,0
pos,5
sense_number,0
type,0
page,0
expression_number,0


In [26]:
# Détail des écarts de `pos` (seul écart attendu) : doit se limiter aux variantes typographiques.
diff_pos = pd.DataFrame({"notebook": mine["pos"], "pipeline": prod["pos"]})
diff_pos = diff_pos[diff_pos["notebook"] != diff_pos["pipeline"]]
print(diff_pos.value_counts().to_frame("lignes"))

                   lignes
notebook pipeline        
n.f      n.f.           5


In [27]:
# Les règles manuelles du notebook et le fichier du pipeline contiennent-elles les mêmes expressions ?
manual_csv = pd.read_csv(ROOT / "data" / "resources" / "expressions_manuelles.csv",
                         dtype=str, keep_default_na=False)
rules_pipeline = {(r.headword, r.definition_principale, r.expression, r.traduction)
                  for r in manual_csv.itertuples()}
rules_notebook = {(head.rstrip("ʾ"), rule["main"], expression, translation)
                  for head, rule in MANUAL_EXPRESSIONS.items()
                  for expression, translation in rule["expressions"]}

print(f"Règles : notebook {len(rules_notebook)} / pipeline {len(rules_pipeline)}")
print("Seulement dans le notebook :", sorted(rules_notebook - rules_pipeline))
print("Seulement dans le pipeline :", sorted(rules_pipeline - rules_notebook))
assert rules_notebook == rules_pipeline

Règles : notebook 77 / pipeline 77
Seulement dans le notebook : []
Seulement dans le pipeline : []


## Bilan et limites

**Résultats.** Le traitement produit 4 909 lignes de lexique et autant de paires
`ar → fr` (9 818 avec le corpus bilingue), identiques à celles du pipeline de référence aux
écarts attendus près (étape 13).

**Limites connues.**

1. **Homographes de rang 3** : 8 vedettes gardent un chiffre (`adda3`). Le pipeline les gère
   par `sense_number`.
2. **Marqueur `ʾ`** : il se confond avec une variante de saisie de la coupure glottale. À ne
   pas réutiliser comme clé de recherche.
3. **Découpage des expressions** : il repose sur 77 expressions validées à la main ; l'extraction
   automatique n'est exacte que pour une partie des cas (étape 7.1). Toute nouvelle entrée avec
   `expr.` doit être ajoutée à `data/resources/expressions_manuelles.csv`.
4. **Doublons** : aucune paire `(src, output)` en double à ce jour (étape 11), mais plusieurs
   sources partagent une même traduction ; à traiter selon l'usage.
5. **Sens principal et expression** : seules les définitions contenant `expr.` sont
   restructurées ; les autres formes d'expressions éventuelles du lexique ne sont pas détectées.

**Reproduire.** Exécuter le notebook depuis la racine du projet ou depuis `notebooks/` ;
vérifier l'empreinte SHA-256 de l'entrée (étape 0). Pour le traitement de production :

```bash
td-arabic build-corpus
```